## Generating vector embeddings for MD files with milvus

### Hybrid model

In [ ]:
import importlib
import vector_db
importlib.reload(vector_db)

<module 'vector_db' from '/home/grzegorz/python/pharmai/vector_db.py'>

In [1]:
from pymilvus import MilvusClient

embedded_milvus = True

if embedded_milvus:
    client = MilvusClient("milvus_db.db")
else:
    client = MilvusClient(host="localhost", port=1953, timeout=10)

collection_name = "CHPL_collection"

In [2]:
from vector_db import (
    initialize_milvus_collection,
    get_existing_ids,
)
initialize_milvus_collection(client, collection_name)

/home/grzegorz/python/pharmai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection 'CHPL_collection' already exists.


In [7]:
import pandas as pd
from pathlib import Path

df_cat = pd.read_csv("Rejestr_Produktow_Leczniczych_z_kategoriami.csv", sep=";", encoding="utf-8")
markdown_folder = Path("charakterystyki_md")
file_paths = list(markdown_folder.glob("*.md"))#[10000:11000]


In [8]:
from vector_db import process_in_batches

process_in_batches(client, file_paths, df_cat, collection_name="CHPL_collection", batch_size=40, pause_time=10)


🔄 Processing batch 1: 40 files
No new data to insert.
⛔ No new data to insert in this batch.

🔄 Processing batch 2: 40 files
No new data to insert.
⛔ No new data to insert in this batch.

🔄 Processing batch 3: 40 files
No new data to insert.
⛔ No new data to insert in this batch.

🔄 Processing batch 4: 40 files
No new data to insert.
⛔ No new data to insert in this batch.

🔄 Processing batch 5: 40 files
No new data to insert.
⛔ No new data to insert in this batch.

🔄 Processing batch 6: 40 files
No new data to insert.
⛔ No new data to insert in this batch.

🔄 Processing batch 7: 40 files
No new data to insert.
⛔ No new data to insert in this batch.

🔄 Processing batch 8: 40 files
No new data to insert.
⛔ No new data to insert in this batch.

🔄 Processing batch 9: 40 files
No new data to insert.
⛔ No new data to insert in this batch.

🔄 Processing batch 10: 40 files
No new data to insert.
⛔ No new data to insert in this batch.

🔄 Processing batch 11: 40 files
No new data to insert.
⛔ N

In [ ]:
results = client.query(
    collection_name=collection_name,
    filter="id >= 0",
    output_fields=["id"]
)

existing_ids = {res["id"] for res in results}

print(f"Found {len(existing_ids)} IDs:")
print(existing_ids)

Found 11275 IDs:
{100106244, 100204550, 100401167, 100401173, 100204567, 100073496, 100466713, 100368412, 100204573, 100466720, 100106280, 100007980, 100368429, 100270127, 100466738, 100401204, 100106296, 100302904, 100204604, 100270140, 100302910, 100335677, 100106304, 100204610, 100335683, 100139083, 100270156, 100401227, 100302927, 100270162, 100401240, 100008027, 100302940, 100008033, 100270179, 100106340, 100434020, 100008040, 100302956, 100270191, 100368501, 100106362, 100335743, 100368518, 100270216, 100302985, 100401291, 100368524, 100270222, 100302991, 100106385, 100368530, 100401300, 100303000, 100434072, 100270239, 100237472, 100368547, 100008100, 100401316, 100368553, 100401322, 100434089, 100434095, 100368560, 100237489, 100499633, 100040887, 100434103, 100434110, 100368576, 100466882, 100073668, 100368582, 100204745, 100466890, 100434126, 100335826, 100434132, 100040918, 100466902, 100335832, 100368613, 100204774, 100434149, 100335849, 100106474, 100434155, 100368620, 100

In [ ]:
from pymilvus import MilvusClient, DataType

schema = MilvusClient.create_schema()

schema.add_field(field_name="id",
                 datatype=DataType.INT64,
                 is_primary=True)
schema.add_field(field_name="text",
                    datatype=DataType.VARCHAR,
                    max_length=65535)
schema.add_field(field_name="dense_vector",
                 datatype=DataType.FLOAT_VECTOR,
                 dim=768)
schema.add_field(field_name="sparse_vector",
                 datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_field(field_name="availability_categories",
                 datatype=DataType.ARRAY,
                 element_type=DataType.VARCHAR,
                 max_length=20,
                 max_capacity=5)

index_params = client.prepare_index_params()
index_params.add_index(field_name="dense_vector",
                       index_type="FLAT",
                       metric_type="COSINE",
                       index_name="dense_index")
index_params.add_index(field_name="sparse_vector",
                       index_type="SPARSE_INVERTED_INDEX",
                       metric_type="IP",
                       index_name="sparse_index")

if client.has_collection(collection_name="hybrid_collection"):
    client.drop_collection(collection_name="hybrid_collection")
client.create_collection(
    collection_name="hybrid_collection",
    schema=schema,
    index_params=index_params)


In [ ]:
res = client.list_indexes(
    collection_name="hybrid_collection"
)

print(res)

['dense_index', 'sparse_index']


In [ ]:
import pandas as pd
df_cat = pd.read_csv("Rejestr_Produktow_Leczniczych_z_kategoriami.csv", sep=";", encoding="utf-8")
df_cat.head(50)
df_cat["Kategorie dostępności"].unique()


array(['Rpz', 'Rp', 'OTC', 'OTC, Rp', 'Lz', 'Lz, Rp', 'Rp, Rpw', 'Rpw',
       'Lz, OTC', 'Rp, Rpz', nan], dtype=object)

### With chunking

In [ ]:
import os
import re
import glob

def extract_categories_and_chunks(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    id_match = re.search(r'char_(\d+)', file_path)
    if not id_match:
        raise ValueError(f"ID not found in file name: {file_path}")

    id_value = id_match.group(1)

    raw_categories = df_cat.loc[df_cat["Identyfikator Produktu Leczniczego"] == id_value]["Kategorie dostępności"]
    if raw_categories.empty:
        categories = []
    else:
        raw_categories = raw_categories.values[0]
        if isinstance(raw_categories, str):
            categories = [cat.strip() for cat in raw_categories.split(",")]
        else:
            categories = []

    content = content[1:].strip()
    chunks = [chunk.strip() for chunk in content.split("\n\n") if chunk.strip()]

    return categories, chunks, id_value

markdown_folder = "charakterystyki_md"
file_paths = glob.glob(os.path.join(markdown_folder, "*.md"))[:7]

all_texts = []
all_categories = []
all_ids = []

for file_path in file_paths:
    categories, chunks, id_value = extract_categories_and_chunks(file_path)
    all_texts.extend(chunks)
    all_categories.extend(categories)
    all_ids.extend(id_value)

### Wtihout chunking

In [ ]:
import os
import re
import glob

def extract_categories_and_text(file_path, df_cat):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    id_match = re.search(r'char_(\d+)', file_path)
    if not id_match:
        raise ValueError(f"ID not found in file name: {file_path}")

    id_value = id_match.group(1)

    try:
        id_value = int(id_value)
    except ValueError:
        raise ValueError(f"ID is not an integer: {id_value}")

    raw_categories = df_cat.loc[df_cat["Identyfikator Produktu Leczniczego"] == id_value]["Kategorie dostępności"]
    if raw_categories.empty:
        categories = []
    else:
        raw_categories = raw_categories.values[0]
        if isinstance(raw_categories, str):
            categories = [cat.strip() for cat in raw_categories.split(",")]
        else:
            categories = []

    return categories, content, id_value

markdown_folder = "charakterystyki_md"
file_paths = glob.glob(os.path.join(markdown_folder, "*.md"))[:20]

all_ids = []
all_texts = []
all_categories = []

for file_path in file_paths:
    categories, text, id_value = extract_categories_and_text(file_path, df_cat)
    all_texts.append(text)
    all_categories.append(categories)
    all_ids.append(id_value)


In [11]:
all_ids

[100132508,
 100376038,
 100280077,
 100021312,
 100459632,
 100081975,
 100211343,
 100371638,
 100478479,
 100425759,
 100072002,
 100452693,
 100433718,
 100100939,
 100248145,
 100330846,
 100366347,
 100198559,
 100111009,
 100145675]

In [ ]:
from pymilvus.model.hybrid import MGTEEmbeddingFunction

mgte_ef = MGTEEmbeddingFunction(
    model_name="Alibaba-NLP/gte-multilingual-base",
)

#vectors = mgte_ef.encode_documents(all_texts)

In [15]:
mgte_ef.dim # (768, 25002)


{'dense': 768, 'sparse': 250002}